# 04 — Evaluation

Evaluates the fine-tuned model on three tiers:

1. **Automated metrics** — grounding compliance, clarification compliance, hallucination detection
2. **LLM-as-judge** — Claude API rates 50 samples on relevance, grounding, dialogue quality, Romanian fluency
3. **Base vs fine-tuned comparison** — head-to-head on 20 test prompts

In [ ]:
import sys, json, re
from pathlib import Path
from collections import defaultdict

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FINE_TUNING_DIR = REPO_ROOT / "fine_tuning"
VAL_FILE        = FINE_TUNING_DIR / "data" / "dataset_val.jsonl"
ADAPTER_DIR     = REPO_ROOT / "fine_tuning" / "adapters" / "qwen_city_advisor_v1"
CITY_INDEX_PATH = FINE_TUNING_DIR / "configs" / "city_index.json"

print(f"Val file   : {VAL_FILE}")
print(f"Adapter    : {ADAPTER_DIR}")

## Part 1 — Automated metrics on the validation set

In [ ]:
from fine_tuning.src.conversation_validator import (
    validate_conversation,
    ValidationError,
    GROUNDING_PHRASES,
    QUESTION_MARK,
    _extract_rag_context,
    _detect_unlisted_cities,
)
from fine_tuning.src.city_loader import load_saved_city_index

city_index = load_saved_city_index(CITY_INDEX_PATH)

# Load validation examples
val_examples = []
for line in VAL_FILE.read_text(encoding="utf-8").splitlines():
    if line.strip():
        val_examples.append(json.loads(line))

print(f"Loaded {len(val_examples)} validation examples")

In [ ]:
# For each val example, reconstruct the raw conversation and run checks

metrics = defaultdict(int)
hallucination_flags = []

for example in val_examples:
    messages = example["messages"]

    # Rebuild raw conversation (skip leading system message = our fixed system prompt)
    conversation = [m for m in messages[1:]]  # skip fixed system prompt
    raw = {"conversation": conversation, "metadata": example.get("metadata", {})}

    # Get assistant turns and context block
    assistant_turns = [m for m in conversation if m["role"] == "assistant"]
    context_block   = _extract_rag_context(conversation)

    metrics["total"] += 1

    # Clarification compliance
    if len(assistant_turns) >= 2:
        clarification_turns = assistant_turns[:-1]
        if any(QUESTION_MARK in t["content"] for t in clarification_turns):
            metrics["has_clarification"] += 1
        else:
            metrics["no_clarification"] += 1
    else:
        metrics["single_turn"] += 1

    # Grounding compliance
    if assistant_turns:
        final = assistant_turns[-1]["content"]
        if GROUNDING_PHRASES.search(final):
            metrics["has_grounding_phrase"] += 1
        else:
            metrics["no_grounding_phrase"] += 1

    # Hallucination check
    if context_block and assistant_turns:
        final = assistant_turns[-1]["content"]
        unlisted = _detect_unlisted_cities(final, context_block, city_index)
        if unlisted:
            hallucination_flags.append({
                "persona": example.get("metadata", {}).get("persona_type"),
                "unlisted_cities": unlisted,
            })
            metrics["possible_hallucination"] += 1
        else:
            metrics["clean_hallucination_check"] += 1

total = metrics["total"]
print(f"\n{'Metric':<35} {'Count':>6}  {'%':>6}")
print("-" * 52)
for key, count in sorted(metrics.items()):
    if key == "total":
        continue
    pct = 100 * count / total
    print(f"  {key:<33} {count:>6}  {pct:>5.1f}%")

if hallucination_flags:
    print(f"\nHallucination flag samples (first 3):")
    for flag in hallucination_flags[:3]:
        print(f"  [{flag['persona']}] unlisted: {flag['unlisted_cities']}")

## Part 2 — LLM-as-judge (50 samples)

Sends 50 random validation examples to Claude for scoring on four dimensions.

In [ ]:
import os, random, time

JUDGE_SAMPLE_SIZE = 50
JUDGE_BACKEND     = "anthropic"   # "anthropic" or "openai"

JUDGE_SYSTEM_PROMPT = """
You are evaluating a Romanian city recommendation chatbot's training data.
Rate the conversation on FOUR dimensions, each 1-5:
1. Relevanță (1-5): Do the final recommendations match the user's stated preferences?
2. Grounding (1-5): Does the final recommendation cite ONLY the [CONTEXT RAG] data, without adding facts?
3. Calitate_dialog (1-5): Is the conversation natural, progressive, and appropriate in length?
4. Limba_romana (1-5): Is the Romanian grammatically correct and natural?

Return ONLY a JSON object: {"relevanta": X, "grounding": X, "calitate_dialog": X, "limba_romana": X}
"""

def format_conversation_for_judge(example):
    lines = []
    for msg in example["messages"]:
        role    = msg["role"].upper()
        content = msg["content"][:500]  # truncate for judge context
        lines.append(f"[{role}]\n{content}")
    return "\n\n".join(lines)

# Sample 50 examples
rng    = random.Random(99)
sample = rng.sample(val_examples, min(JUDGE_SAMPLE_SIZE, len(val_examples)))

print(f"Will judge {len(sample)} examples using {JUDGE_BACKEND} API")

In [ ]:
from fine_tuning.src.api_client import make_client

judge_client = make_client(backend=JUDGE_BACKEND, api_key=os.getenv("ANTHROPIC_API_KEY") or os.getenv("OPENAI_API_KEY"))

scores = []
errors = 0

for idx, example in enumerate(sample, start=1):
    try:
        conv_text   = format_conversation_for_judge(example)
        result      = judge_client.generate(JUDGE_SYSTEM_PROMPT, conv_text)
        result["persona_type"] = example.get("metadata", {}).get("persona_type")
        scores.append(result)
        if idx % 10 == 0:
            print(f"  Judged {idx}/{len(sample)}...")
    except Exception as exc:
        print(f"  [{idx}] Judge error: {exc}")
        errors += 1
    time.sleep(1.2)

print(f"\nJudged {len(scores)}/{len(sample)} (errors: {errors})")

In [ ]:
# Aggregate scores
dims = ["relevanta", "grounding", "calitate_dialog", "limba_romana"]

if scores:
    print(f"\n{'Dimension':<20}  {'Avg':>5}  {'Min':>5}  {'Max':>5}")
    print("-" * 40)
    for dim in dims:
        vals = [s[dim] for s in scores if isinstance(s.get(dim), (int, float))]
        if vals:
            print(f"  {dim:<18}  {sum(vals)/len(vals):>5.2f}  {min(vals):>5}  {max(vals):>5}")
    overall = [sum(s.get(d, 0) for d in dims) / len(dims) for s in scores]
    print(f"\n  Overall average: {sum(overall)/len(overall):.2f} / 5.0")

## Part 3 — Model inference check (if adapter is available)

In [ ]:
# Only runs if the fine-tuned adapter exists
if not ADAPTER_DIR.exists():
    print(f"Adapter not found at {ADAPTER_DIR}. Run 03_fine_tune.ipynb first.")
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel
    from fine_tuning.configs.lora_config import DEFAULT_MODEL_ID
    from fine_tuning.src.dataset_formatter import SYSTEM_PROMPT

    print(f"Loading fine-tuned model from {ADAPTER_DIR}...")

    tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)

    base_model = AutoModelForCausalLM.from_pretrained(
        DEFAULT_MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    ft_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
    ft_model.eval()

    def infer(messages, max_new=300):
        text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(ft_model.device)
        with torch.no_grad():
            out = ft_model.generate(**inputs, max_new_tokens=max_new, temperature=0.3,
                                    do_sample=True, pad_token_id=tokenizer.eos_token_id)
        new = out[0][inputs["input_ids"].shape[1]:]
        return tokenizer.decode(new, skip_special_tokens=True)

    # Test: no context — should ask questions, NOT mention specific cities
    test_no_ctx = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "Bună! Vreau să mă mut. Am un câine și lucrez în IT."},
    ]
    resp_no_ctx = infer(test_no_ctx)
    print("\n[No context — should ask questions]")
    print(resp_no_ctx)

    # Check: no city names hallucinated
    known_cities = [c["display_name"] for c in city_index.values()]
    hallucinated = [city for city in known_cities if city.lower() in resp_no_ctx.lower()]
    print(f"\nCity names mentioned without context: {hallucinated or 'none — good!'}")
    print(f"Contains '?' (clarification question): {'?' in resp_no_ctx}")